# Local translation review workflow

This notebook now calls the reviewed local package instead of executing the inherited platform-specific experiment. It is a deliberate replacement, not behavioral equivalence or reproduction of historical scores. The original source remains in unchanged Git history. Run from the repository root after installing the locked package with its needed extras.

No install, credentials, uploads, downloads, model load or training runs by default. Synthetic examples are in `scripts/demo.py`; they do not establish real-data quality. This notebook does not start that demo automatically either. No dataset/model is bundled or licensed by this replacement.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print(json.dumps({"mode": "local-only", "training_enabled": False}))


## Explicit local input check
Set `NOTEBOOK_CHECK_DATA=1` and the listed local path settings to run the package check. Missing settings or paths fail rather than falling back to a download. Exact annotation, patient/subject/group partitions and rights still require review; this is not a quality benchmark.

PARALLEL_CSV


In [ ]:
if os.environ.get("NOTEBOOK_CHECK_DATA") == "1":
    settings = [('--data', 'PARALLEL_CSV')]
    argv = [sys.executable, "-m", "local_translation", "check-data"]
    for flag, name in settings:
        value = os.environ.get(name)
        if not value or not Path(value).exists():
            raise ValueError("Supply an existing permitted local path for " + name)
        argv.extend([flag, str(Path(value).resolve())])
    completed = subprocess.run(argv, check=True, capture_output=True, text=True)
    print(completed.stdout)
else:
    print("Local input check skipped; no data or model was loaded.")


## Optional explicit CPU training
Set `NOTEBOOK_RUN_TRAINING=1` only after reviewing the data and costs on your own machine. The output setting must name a new directory. This starts real training, not a preview. One epoch is a wiring default, not a finished model. Default model dimensions can be slow on CPU. Run the project's tiny standalone demo first if you only need a bounded engineering check. No online model fallback is provided.

Required settings: PARALLEL_CSV, TRANSLATION_BASE_MODEL, TRANSLATION_OUTPUT. Refer to the package CLI for architecture/batch options. All artifact files must be trusted. Never point a pickle-based checkpoint loader at untrusted weights.


In [ ]:
if os.environ.get("NOTEBOOK_RUN_TRAINING") == "1":
    data_value = os.environ.get("PARALLEL_CSV")
    output_value = os.environ.get("TRANSLATION_OUTPUT")
    if not data_value or not Path(data_value).exists():
        raise ValueError("Supply existing permitted local training data.")
    if not output_value or Path(output_value).exists():
        raise ValueError("Supply a new explicit output directory.")
    data_path = str(Path(data_value).resolve())
    output_path = str(Path(output_value).resolve())
    base_value = os.environ.get("TRANSLATION_BASE_MODEL")
    if not base_value or not Path(base_value).is_dir():
        raise ValueError("Supply an existing trusted local base-model directory.")
    base_path = str(Path(base_value).resolve())
    argv = [sys.executable, "-m", 'local_translation', 'train', base_path, data_path, output_path, '--epochs', '1', '--batch-size', '1']
    completed = subprocess.run(argv, check=True, capture_output=True, text=True)
    print(completed.stdout)
else:
    print("Training skipped; no artifact is created.")


## Limits
The WMT14 download and original model-hub fetch are removed. The local model must have the intended language direction. No WMT score, pretrained translation quality, BLEU/chrF or corpus-rights clearance is asserted.
